# Notebook 6 — Final 38-Task Evaluation

This notebook is the **final end-to-end evaluation tracker** for the 34-lecture MIT 18.06 Tutor system.

It uses the frozen system configuration:

- Lectures: **1–34**
- Chunk size: **1200**
- Retrieval `k`: **8**
- Reranker: **ON/Top 3**

The 38 tasks are maintained in the Excel evaluation set rather than duplicated inside this notebook.

### Evaluation workflow

1. Run each task manually in the final web app.
2. Let LangSmith capture the technical trace automatically.
3. Record `Pass` or `Fail`, a short note, failure type if needed, and optionally the LangSmith Run ID.
4. Do **not** tune the RAG configuration during this evaluation.
5. Analyze pass rate and failure patterns after the test set is complete.


## Step 0 — Setup

Put the updated Excel question set in:

`AI_Engineering_Final_Project/evaluation/Lecture_Tutor_38_Task_Evaluation_Set_UPDATED.xlsx`

The notebook will load the 38 tasks directly from that file.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path('/content/drive/MyDrive/AI_Engineering_Final_Project')
EVALUATION_DIR = PROJECT_ROOT / 'evaluation'
EVALUATION_DIR.mkdir(parents=True, exist_ok=True)

QUESTION_SET_PATH = (
    EVALUATION_DIR
    / 'Lecture_Tutor_38_Task_Evaluation_Set_UPDATED.xlsx'
)

RESULTS_PATH = (
    EVALUATION_DIR
    / 'Lecture_Tutor_38_Task_Final_Results.xlsx'
)

FINAL_CONFIG = {
    'lecture_start': 1,
    'lecture_end': 34,
    'chunk_size': 1200,
    "retrieval_k": 8,
    "rerank_top_n": 3,
    "reranker": True,
    "llm": "gpt-5.6-luna",
    "reasoning_effort": "high",
}

print('Question set:', QUESTION_SET_PATH)
print('Results file:', RESULTS_PATH)
print('Frozen config:', FINAL_CONFIG)


## Step 1 — Load the Exact 38-Task Evaluation Set

The Excel file is the single source of truth for the task wording, expected tools, and pass criteria.


In [ ]:
assert QUESTION_SET_PATH.exists(), (
    f'Question set not found: {QUESTION_SET_PATH}\n'
    'Upload the updated Excel file to the evaluation folder first.'
)

evaluation_df = pd.read_excel(
    QUESTION_SET_PATH,
    sheet_name='Eval Set',
    header=3,       # row 4 contains the real column headers
)

# Remove completely blank rows, if any.
evaluation_df = evaluation_df.dropna(how='all').copy()

# Keep only the actual task rows.
evaluation_df = evaluation_df[
    pd.to_numeric(evaluation_df['ID'], errors='coerce').notna()
].copy()

evaluation_df['ID'] = evaluation_df['ID'].astype(int)

print('Tasks loaded:', len(evaluation_df))
display(evaluation_df.head(8))


Tasks loaded: 38


,ID,Category,Lecture Scope,Task / Prompt (turns separated by >>),Expected Tool(s) Called,Pass Criteria,Result (Pass/Fail),Failure Type,Notes,LangSmith Run ID
0,1,Direct Concept,All,What is the column space of a matrix?,search_lecture (no lecture_number),Defines column space correctly as the span of ...,NaN,NaN,Use Pass only when the core behavior succeeds....,NaN
1,2,Direct Concept,All,What does it mean for a matrix to be invertible?,search_lecture (no lecture_number),Correctly explains invertibility (existence of...,NaN,NaN,NaN,NaN
2,3,Direct Concept,All,What is an eigenvector?,search_lecture (no lecture_number),Correctly defines an eigenvector using Ax = λx...,NaN,NaN,NaN,NaN
3,4,Direct Concept,All,What is the difference between the null space ...,search_lecture (no lecture_number),Correctly distinguishes null space (solutions ...,NaN,NaN,NaN,NaN
4,5,Direct Concept,All,What is a positive definite matrix?,search_lecture (no lecture_number),Correctly explains positive definiteness using...,NaN,NaN,NaN,NaN
5,6,Direct Concept,All,What does it mean for two vectors to be orthog...,search_lecture (no lecture_number),Correctly explains orthogonality using zero do...,NaN,NaN,NaN,NaN
6,7,Lecture-Specific,L4,"In Lecture 4, why does A = LU matter?",search_lecture(lecture_number=4),Answer is scoped to Lecture 4 only; explains L...,NaN,NaN,NaN,NaN
7,8,Lecture-Specific,L12,"In Lecture 12, how are incidence matrices used...",search_lecture(lecture_number=12),Answer scoped to Lecture 12; correctly connect...,NaN,NaN,NaN,NaN


## Step 2 — Validate the Final Test Set

This prevents accidental evaluation with the wrong spreadsheet or a partially edited file.


In [ ]:
EXPECTED_TASK_COUNT = 38

EXPECTED_CATEGORY_COUNTS = {
    'Direct Concept': 6,
    'Lecture-Specific': 6,
    'Comparison': 5,
    'Memory / Follow-up': 5,
    'Out-of-Scope': 4,
    'Quiz Gen/Grading': 4,
    'Weak-Topic Tracking': 3,
    'Course Builder': 3,
    'Source/Timestamp Check': 2,
}

assert len(evaluation_df) == EXPECTED_TASK_COUNT, (
    f'Expected {EXPECTED_TASK_COUNT} tasks, found {len(evaluation_df)}.'
)

actual_counts = (
    evaluation_df['Category']
    .value_counts()
    .to_dict()
)

assert actual_counts == EXPECTED_CATEGORY_COUNTS, (
    'Category counts do not match the frozen 38-task set.\n'
    f'Expected: {EXPECTED_CATEGORY_COUNTS}\n'
    f'Found: {actual_counts}'
)

required_columns = [
    'ID',
    'Category',
    'Lecture Scope',
    'Task / Prompt (turns separated by >>)',
    'Expected Tool(s) Called',
    'Pass Criteria',
    'Result (Pass/Fail)',
    'Failure Type',
    'Notes',
    'LangSmith Run ID',
]

missing_columns = [
    col for col in required_columns
    if col not in evaluation_df.columns
]

assert not missing_columns, (
    f'Missing expected columns: {missing_columns}'
)

print('✅ Exact 38-task evaluation set validated.')
display(
    evaluation_df.groupby('Category')
    .size()
    .rename('Tasks')
    .reset_index()
)


✅ Exact 38-task evaluation set validated.


,Category,Tasks
0,Comparison,5
1,Course Builder,3
2,Direct Concept,6
3,Lecture-Specific,6
4,Memory / Follow-up,5
5,Out-of-Scope,4
6,Quiz Gen/Grading,4
7,Source/Timestamp Check,2
8,Weak-Topic Tracking,3


## Step 3 — Review the Test Set Before Running

Use this to inspect the complete task list without changing anything.


In [ ]:
display(
    evaluation_df[
        [
            'ID',
            'Category',
            'Lecture Scope',
            'Task / Prompt (turns separated by >>)',
            'Pass Criteria',
        ]
    ]
)


,ID,Category,Lecture Scope,Task / Prompt (turns separated by >>),Pass Criteria
0,1,Direct Concept,All,What is the column space of a matrix?,Defines column space correctly as the span of ...
1,2,Direct Concept,All,What does it mean for a matrix to be invertible?,Correctly explains invertibility (existence of...
2,3,Direct Concept,All,What is an eigenvector?,Correctly defines an eigenvector using Ax = λx...
3,4,Direct Concept,All,What is the difference between the null space ...,Correctly distinguishes null space (solutions ...
4,5,Direct Concept,All,What is a positive definite matrix?,Correctly explains positive definiteness using...
5,6,Direct Concept,All,What does it mean for two vectors to be orthog...,Correctly explains orthogonality using zero do...
6,7,Lecture-Specific,L4,"In Lecture 4, why does A = LU matter?",Answer is scoped to Lecture 4 only; explains L...
7,8,Lecture-Specific,L12,"In Lecture 12, how are incidence matrices used...",Answer scoped to Lecture 12; correctly connect...
8,9,Lecture-Specific,L17,"In Lecture 17, what does Gram-Schmidt actually...",Answer scoped to Lecture 17; explains orthogon...
9,10,Lecture-Specific,L21,"In Lecture 21, how do you find the eigenvalues...",Answer scoped to Lecture 21; references det(A ...


## Step 4 — Helper: Show One Test

Use `show_test(1)`, `show_test(18)`, etc. while running the web app manually.


In [ ]:
def show_test(test_id):
    row = evaluation_df.loc[evaluation_df['ID'] == int(test_id)]

    if row.empty:
        print(f'Test ID {test_id} not found.')
        return

    row = row.iloc[0]

    print('=' * 90)
    print(f"TEST {row['ID']} — {row['Category']}")
    print(f"Lecture scope: {row['Lecture Scope']}")
    print('=' * 90)

    print('\nTASK / PROMPT')
    print(row['Task / Prompt (turns separated by >>)'])

    print('\nEXPECTED TOOL BEHAVIOR')
    print(row['Expected Tool(s) Called'])

    print('\nPASS CRITERIA')
    print(row['Pass Criteria'])

    print('\n' + '=' * 90)


# Example:
show_test(1)


TEST 1 — Direct Concept
Lecture scope: All

TASK / PROMPT
What is the column space of a matrix?

EXPECTED TOOL BEHAVIOR
search_lecture (no lecture_number)

PASS CRITERIA
Defines column space correctly as the span of a matrix's columns; answer is grounded in relevant MIT 18.06 lecture evidence and includes a valid lecture/source timestamp. Do not require one exact lecture unless the prompt names it.



## Step 5 — Record a Result

After running a task in the **final web app**, record the judgment here.

Use only:

- `result="Pass"` when the core behavior succeeds.
- `result="Fail"` when there is a meaningful defect.

For a failure, use one of these labels when possible:

`retrieval`, `wrong_lecture`, `grounding`, `memory`, `scope`, `quiz`, `state_tracking`, `course_builder`, `source_timestamp`, `ui`, `other`


In [ ]:
ALLOWED_RESULTS = {"Pass", "Fail"}

ALLOWED_FAILURE_TYPES = {
    "",
    "retrieval",
    "wrong_lecture",
    "grounding",
    "memory",
    "scope",
    "quiz",
    "state_tracking",
    "course_builder",
    "source_timestamp",
    "ui",
    "other",
}


def record_result(
    test_id,
    result,
    notes="",
    failure_type="",
    langsmith_run_id="",
):
    test_id = int(test_id)

    assert result in ALLOWED_RESULTS, (
        f"result must be one of {sorted(ALLOWED_RESULTS)}"
    )

    assert failure_type in ALLOWED_FAILURE_TYPES, (
        f"Unknown failure type: {failure_type}"
    )

    if result == "Pass":
        failure_type = ""

    mask = evaluation_df["ID"] == test_id

    if not mask.any():
        raise ValueError(f"Test ID {test_id} not found.")

    evaluation_df.loc[mask, "Result (Pass/Fail)"] = result
    evaluation_df.loc[mask, "Failure Type"] = failure_type
    evaluation_df.loc[mask, "Notes"] = notes
    evaluation_df.loc[mask, "LangSmith Run ID"] = langsmith_run_id

    print(f"✅ Recorded Test {test_id}: {result}")

    # Show exactly what was recorded
    display(
        evaluation_df.loc[
            mask,
            [
                "ID",
                "Category",
                "Result (Pass/Fail)",
                "Failure Type",
                "Notes",
                "LangSmith Run ID",
            ],
        ]
    )


# ============================================================
# RECORD YOUR CURRENT TEST HERE
# Change these values for each test, then run this cell.
# ============================================================

record_result(
    test_id=38,
    result="Pass",
    notes="All cited sources are from Lecture 14 only (no cross-lecture leakage).",
    failure_type="",
    langsmith_run_id="01a0689b-5420-7e61-aeee-8c0c6f82278c",
)



✅ Recorded Test 38: Pass


,ID,Category,Result (Pass/Fail),Failure Type,Notes,LangSmith Run ID
37,38,Source/Timestamp Check,Pass,,All cited sources are from Lecture 14 only (no...,01a0689b-5420-7e61-aeee-8c0c6f82278c


## Step 6 — Show the Next Unrun Test

This makes it easier to work through all 38 tests in order.


In [ ]:
def next_unrun_test():
    result_col = evaluation_df['Result (Pass/Fail)'].fillna('').astype(str).str.strip()

    remaining = evaluation_df[
        ~result_col.isin(['Pass', 'Fail'])
    ]

    if remaining.empty:
        print('✅ All 38 tests have been completed.')
        return None

    next_id = int(remaining.iloc[0]['ID'])
    show_test(next_id)
    return next_id


next_unrun_test()


✅ All 38 tests have been completed.


## Step 7 — Save Working Results

This writes a separate results workbook. The original question set remains unchanged.


In [ ]:
def save_results():
    # Current detailed results
    results_df = evaluation_df.copy()

    # Completed-only category summary
    completed = results_df[
        results_df['Result (Pass/Fail)'].isin(['Pass', 'Fail'])
    ].copy()

    if completed.empty:
        summary_df = pd.DataFrame(
            columns=[
                'Category',
                'Total Tasks',
                'Completed',
                'Passed',
                'Failed',
                'Pass Rate',
            ]
        )
    else:
        total_by_category = (
            results_df.groupby('Category')
            .size()
            .rename('Total Tasks')
        )

        completed_count = (
            completed.groupby('Category')
            .size()
            .rename('Completed')
        )

        passed_count = (
            completed[completed['Result (Pass/Fail)'] == 'Pass']
            .groupby('Category')
            .size()
            .rename('Passed')
        )

        failed_count = (
            completed[completed['Result (Pass/Fail)'] == 'Fail']
            .groupby('Category')
            .size()
            .rename('Failed')
        )

        summary_df = pd.concat(
            [
                total_by_category,
                completed_count,
                passed_count,
                failed_count,
            ],
            axis=1,
        ).fillna(0)

        summary_df[['Total Tasks', 'Completed', 'Passed', 'Failed']] = (
            summary_df[
                ['Total Tasks', 'Completed', 'Passed', 'Failed']
            ]
            .astype(int)
        )

        summary_df['Pass Rate'] = np.where(
            summary_df['Completed'] > 0,
            summary_df['Passed'] / summary_df['Completed'],
            np.nan,
        )

        summary_df = summary_df.reset_index()

    with pd.ExcelWriter(RESULTS_PATH, engine='openpyxl') as writer:
        results_df.to_excel(
            writer,
            sheet_name='Evaluation Results',
            index=False,
        )
        summary_df.to_excel(
            writer,
            sheet_name='Summary',
            index=False,
        )

    print('Saved:', RESULTS_PATH)


save_results()


## Step 8 — Progress Summary

Pass rate is calculated using **completed tests only**, so unfinished tasks do not count as failures.


In [ ]:
def evaluation_summary():
    completed = evaluation_df[
        evaluation_df['Result (Pass/Fail)'].isin(['Pass', 'Fail'])
    ].copy()

    total = len(evaluation_df)
    completed_n = len(completed)
    passed_n = int((completed['Result (Pass/Fail)'] == 'Pass').sum())
    failed_n = int((completed['Result (Pass/Fail)'] == 'Fail').sum())

    print(f'Total tasks:     {total}')
    print(f'Completed:       {completed_n}')
    print(f'Remaining:       {total - completed_n}')
    print(f'Passed:          {passed_n}')
    print(f'Failed:          {failed_n}')

    if completed_n:
        print(f'Pass rate:       {passed_n / completed_n:.1%}')
    else:
        print('Pass rate:       not available yet')

    category_rows = []

    for category, group in evaluation_df.groupby('Category', sort=False):
        done = group[
            group['Result (Pass/Fail)'].isin(['Pass', 'Fail'])
        ]

        passed = int(
            (done['Result (Pass/Fail)'] == 'Pass').sum()
        )
        failed = int(
            (done['Result (Pass/Fail)'] == 'Fail').sum()
        )

        category_rows.append({
            'Category': category,
            'Total Tasks': len(group),
            'Completed': len(done),
            'Passed': passed,
            'Failed': failed,
            'Pass Rate': (
                passed / len(done)
                if len(done)
                else np.nan
            ),
        })

    summary = pd.DataFrame(category_rows)
    display(summary)


evaluation_summary()


Total tasks:     38
Completed:       38
Remaining:       0
Passed:          32
Failed:          6
Pass rate:       84.2%


,Category,Total Tasks,Completed,Passed,Failed,Pass Rate
0,Direct Concept,6,6,6,0,1.000000
1,Lecture-Specific,6,6,5,1,0.833333
2,Comparison,5,5,4,1,0.800000
3,Memory / Follow-up,5,5,3,2,0.600000
4,Out-of-Scope,4,4,4,0,1.000000
5,Quiz Gen/Grading,4,4,3,1,0.750000
6,Weak-Topic Tracking,3,3,3,0,1.000000
7,Course Builder,3,3,2,1,0.666667
8,Source/Timestamp Check,2,2,2,0,1.000000


## Step 9 — Failure Analysis

Do not immediately fix the system during the final evaluation.

First collect the failures, then look for repeated patterns.


In [ ]:
failed_df = evaluation_df[
    evaluation_df['Result (Pass/Fail)'] == 'Fail'
].copy()

if failed_df.empty:
    print('No failed tests recorded yet.')
else:
    display(
        failed_df[
            [
                'ID',
                'Category',
                'Lecture Scope',
                'Task / Prompt (turns separated by >>)',
                'Failure Type',
                'Notes',
                'LangSmith Run ID',
            ]
        ]
    )

    print('\nFailure type counts:')
    display(
        failed_df['Failure Type']
        .fillna('')
        .replace('', 'unclassified')
        .value_counts()
        .rename_axis('Failure Type')
        .reset_index(name='Count')
    )


,ID,Category,Lecture Scope,Task / Prompt (turns separated by >>),Failure Type,Notes,LangSmith Run ID
11,12,Lecture-Specific,L33,"In Lecture 33, what's the difference between a...",scope,Pseudoinverse omitted despite being required b...,01a06848-f3f1-7b70-b07a-c2348585914b
14,15,Comparison,L21 vs L29,When would you use eigenvalue decomposition ve...,wrong_lecture,"Cites L22/L23, not foundational L21 source.",01a06853-eead-7a30-83ab-f4459e09923e
20,21,Memory / Follow-up,L24,Turn 1: 'What is a Markov matrix?'\n>> Turn 2:...,other,"Transient syntax error; resend succeeded, cont...",
21,22,Memory / Follow-up,L18–20,Turn 1: 'Quiz me on determinants.'\n>> Turn 2 ...,other,SyntaxError: The string did not match the expe...,
29,30,Quiz Gen/Grading,All,Turn 1: 'Quiz me on determinants.'\n>> Turn 2:...,grounding,Silently graded ambiguous answer instead of as...,
35,36,Course Builder,Auto (from weak topic),After missing a quiz question earlier in the s...,state_tracking,"Fell back to L31, not the actual first recorde...",01a06894-2a4f-7702-8829-c40401315c88



Failure type counts:


,Failure Type,Count
0,other,2
1,scope,1
2,wrong_lecture,1
3,grounding,1
4,state_tracking,1


## Step 10 — Final Report Numbers

Run this after all 38 tasks are complete. These are the numbers to use in the final report/presentation.


In [ ]:
completed = evaluation_df[
    evaluation_df['Result (Pass/Fail)'].isin(['Pass', 'Fail'])
].copy()

if len(completed) != 38:
    print(
        f'Final evaluation is not complete yet: '
        f'{len(completed)}/38 tests recorded.'
    )
else:
    passed = int(
        (completed['Result (Pass/Fail)'] == 'Pass').sum()
    )
    failed = int(
        (completed['Result (Pass/Fail)'] == 'Fail').sum()
    )

    print('FINAL 38-TASK EVALUATION')
    print('------------------------')
    print(f'Passed: {passed}/38')
    print(f'Failed: {failed}/38')
    print(f'Overall pass rate: {passed / 38:.1%}')

    print('\nFailures by type:')
    display(
        completed[
            completed['Result (Pass/Fail)'] == 'Fail'
        ]['Failure Type']
        .fillna('')
        .replace('', 'unclassified')
        .value_counts()
        .rename_axis('Failure Type')
        .reset_index(name='Count')
    )


FINAL 38-TASK EVALUATION
------------------------
Passed: 32/38
Failed: 6/38
Overall pass rate: 84.2%

Failures by type:


,Failure Type,Count
0,other,2
1,scope,1
2,wrong_lecture,1
3,grounding,1
4,state_tracking,1
